# DE1 — Final Project Notebook> Professeur : Badr TAJINI - Data Engineering I - ESIEE 2025-2026---> Étudiants : DIALLO Samba & DIOP Mouhamed---Ceci est l’artéfact exécutable principal. Remplir la config, exécuter la baseline puis la pipeline optimisée, et enregistrer les preuves.

## 0. Charger la configuration

In [ ]:
import yaml, pathlib, datetimefrom pyspark.sql import SparkSession, functions as F, types as Twith open("de1_project_config.yml") as f:    CFG = yaml.safe_load(f)spark = (SparkSession.builder    .appName("de1-lakehouse-project")    .config("spark.sql.adaptive.enabled", "true")    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")    .config("spark.driver.memory", "4g")    .config("spark.executor.memory", "4g")    .config("spark.memory.fraction", "0.6")    .config("spark.memory.storageFraction", "0.3")    .getOrCreate())print(f"Spark version: {spark.version}")print(f"Projet: {CFG['project']['name']}")CFG

## 1. Bronze — import des données brutes

In [ ]:
raw_glob = CFG["paths"]["raw_csv_glob"]bronze = CFG["paths"]["bronze"]proof = CFG["paths"]["proof"]df_raw = (spark.read    .option("header", "false")    .option("inferSchema", "false")    .option("delimiter", "\t")    .csv(raw_glob)    .toDF("prev", "curr", "type", "n"))row_count = df_raw.count()df_raw.write.mode("overwrite").csv(bronze)print(f"Bronze écrit : {bronze}, lignes : {row_count:,}")

## 2. Silver — nettoyage et typage

In [ ]:
silver = CFG["paths"]["silver"]df_silver = (df_raw    .withColumn("n", F.col("n").cast("integer"))    .filter(F.col("n").isNotNull())    .filter(F.col("n") >= 0)    .filter(F.length(F.col("curr")) > 0)    .dropDuplicates())silver_count = df_silver.count()df_silver.write.mode("overwrite").parquet(silver)print(f"Silver écrit : {silver}, lignes : {silver_count:,}")

## 3. Gold — tables analytiques

In [ ]:
gold = CFG["paths"]["gold"]queries = CFG["queries"]pathlib.Path(gold).mkdir(parents=True, exist_ok=True)df_silver.createOrReplaceTempView("silver")df_q1 = spark.sql(queries["q1"]["sql"])q1_count = df_q1.count()df_q1.write.mode("overwrite").parquet(f"{gold}/q1_daily_aggregation")print(f"Q1 écrit, lignes : {q1_count:,}")df_q2 = spark.sql(queries["q2"]["sql"])q2_count = df_q2.count()df_q2.write.mode("overwrite").parquet(f"{gold}/q2_top_referrers")print(f"Q2 écrit, lignes : {q2_count:,}")df_q3 = spark.sql(queries["q3"]["sql"])q3_count = df_q3.count()df_q3.write.mode("overwrite").parquet(f"{gold}/q3_filtered_analysis")print(f"Q3 écrit, lignes : {q3_count:,}")print(f"Gold écrit : {gold}")

## 4. Plans de base et métriques

In [ ]:
pathlib.Path(proof).mkdir(parents=True, exist_ok=True)df_q1_baseline = spark.sql(queries["q1"]["sql"])plan_q1 = df_q1_baseline._jdf.queryExecution().executedPlan().toString()with open(f"{proof}/baseline_q1_plan.txt", "w") as f:    f.write(str(datetime.datetime.now()) + "\n")    f.write(plan_q1)df_q2_baseline = spark.sql(queries["q2"]["sql"])plan_q2 = df_q2_baseline._jdf.queryExecution().executedPlan().toString()with open(f"{proof}/baseline_q2_plan.txt", "w") as f:    f.write(str(datetime.datetime.now()) + "\n")    f.write(plan_q2)df_q3_baseline = spark.sql(queries["q3"]["sql"])plan_q3 = df_q3_baseline._jdf.queryExecution().executedPlan().toString()with open(f"{proof}/baseline_q3_plan.txt", "w") as f:    f.write(str(datetime.datetime.now()) + "\n")    f.write(plan_q3)print("Plans de base sauvegardés. Relever maintenant les métriques Spark UI.")

## 5. Optimisation — layout et jointures

In [ ]:
layout = CFG["layout"]target_size_mb = layout["target_file_size_mb"]df_silver_reloaded = spark.read.parquet(silver)total_bytes = df_silver_reloaded.count() * 100num_partitions = max(4, int(total_bytes / (target_size_mb * 1024 * 1024)))df_silver_opt = (df_silver_reloaded    .repartition(num_partitions)    .sortWithinPartitions(F.desc("n")))silver_opt = f"{silver}_optimized"df_silver_opt.write.mode("overwrite").parquet(silver_opt)print(f"Silver optimisé écrit : {silver_opt}")df_silver_opt.createOrReplaceTempView("silver")df_q1_opt = spark.sql(queries["q1"]["sql"])plan_q1_opt = df_q1_opt._jdf.queryExecution().executedPlan().toString()with open(f"{proof}/optimized_q1_plan.txt", "w") as f:    f.write(str(datetime.datetime.now()) + "\n")    f.write(f"Optimisations : {num_partitions} partitions, trié par n desc\n")    f.write(plan_q1_opt)df_q2_opt = spark.sql(queries["q2"]["sql"])plan_q2_opt = df_q2_opt._jdf.queryExecution().executedPlan().toString()with open(f"{proof}/optimized_q2_plan.txt", "w") as f:    f.write(str(datetime.datetime.now()) + "\n")    f.write(f"Optimisations : {num_partitions} partitions, trié par n desc\n")    f.write(plan_q2_opt)df_q3_opt = spark.sql(queries["q3"]["sql"])plan_q3_opt = df_q3_opt._jdf.queryExecution().executedPlan().toString()with open(f"{proof}/optimized_q3_plan.txt", "w") as f:    f.write(str(datetime.datetime.now()) + "\n")    f.write(f"Optimisations : {num_partitions} partitions, trié par n desc\n")    f.write(plan_q3_opt)print("Plans optimisés sauvegardés. Relever maintenant les métriques Spark UI.")

## 6. Nettoyage

In [ ]:
spark.stop()print("Session Spark arrêtée.")